# Atividade: REINFORCE — programe do zero seu primeiro algoritmo de Deep RL com PyTorch 🤖

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/thumbnail.jpg" alt="Unit 4 Thumbnail">

Neste notebook, **você programará do zero o algoritmo REINFORCE** (Policy Gradient / Monte Carlo) usando PyTorch e treinará um agente para jogar **CartPole-v1** 🎯.

Este notebook é uma simplificação do unit4.ipynb do **Curso de Aprendizagem por Reforço Profundo** (Deep Reinforcement Learning) https://huggingface.co/datasets/huggingface-deep-rl-course/.

> ⚠️ **Versão adaptada para as versões atuais do gymnasium (≥ 1.0)**: o original usava a API antiga do gym (`env.reset()` sem tupla, `env.step` com 4 valores). Aqui os códigos já usam a API nova, e a gravação de vídeo usa `render_mode="rgb_array"`.

⬇️ Aqui está um exemplo do que **você vai alcançar** ⬇️

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/reinforce-CartPole-v1/resolve/main/replay.mp4" type="video/mp4"></video>

### 🎮 Ambiente:

- [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) — equilibrar uma vareta em cima de um carrinho: empurrando para a esquerda ou para a direita.

### 📚 Biblioteca:

- [PyTorch](https://pytorch.org/) — o REINFORCE é implementado do zero, sem biblioteca de RL pronta.

## Objetivos deste notebook 🏆
No final do notebook, você irá:
- Compreender como funciona um **Policy Gradient** (REINFORCE).
- Implementá-lo **do zero** com PyTorch e treinar um agente no CartPole-v1.
- Avaliar o agente e **gravar um vídeo de replay** do resultado.

## Instalar as dependências 🔽

O PyTorch e o matplotlib já vêm instalados no Colab. Precisamos apenas do **gymnasium** (ambientes) e do **imageio** (gravação do vídeo de replay).

In [ ]:
%pip install -q "gymnasium[classic-control]" imageio imageio-ffmpeg

# - gymnasium[classic-control]: ambientes clássicos (CartPole) + pygame (renderização)
# - imageio + imageio-ffmpeg: gravação do vídeo de replay (record_video)

## Importar os pacotes 📦

Além das bibliotecas instaladas, importamos:
- `deque`: fila para calcular as pontuações médias durante o treino
- `Categorical`: distribuição de probabilidades sobre as ações (a saída da nossa política)
- `imageio`: geração do vídeo de replay

In [ ]:
import numpy as np
from collections import deque

import matplotlib.pyplot as plt
%matplotlib inline

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

# Gymnasium
import gymnasium as gym

import imageio

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

# Primeiro agente: jogando CartPole-v1 🤖

## Criar o ambiente e entendê-lo 🎮

O CartPole-v1 tem:
- **4 observações** (posição e velocidade do carrinho, ângulo e velocidade angular da vareta)
- **2 ações** (0 = empurrar para a esquerda, 1 = para a direita)
- Recompensa **+1 a cada passo** com a vareta de pé; o episódio termina quando ela cai ou após 500 passos.

É um ambiente simples de propósito: assim, se algo der errado no treino, sabemos que o problema está na **nossa implementação**, não na complexidade do ambiente.

In [ ]:
env_id = "CartPole-v1"

# Cria o ambiente de treino e o de avaliação
env = gym.make(env_id)
eval_env = gym.make(env_id)

# Tamanhos dos espaços de observação e de ação
s_size = env.observation_space.shape[0]
a_size = env.action_space.n

In [ ]:
print("_____ESPAÇO DE OBSERVAÇÃO_____\n")
print("Dimensão do estado:", s_size)
print("Exemplo de observação:", env.observation_space.sample())

print("\n_____ESPAÇO DE AÇÃO_____\n")
print("Número de ações:", a_size)
print("Exemplo de ação:", env.action_space.sample())

## A arquitetura do REINFORCE 🧠

Nossa política é uma **rede neural com duas camadas totalmente conectadas**:

- `fc1`: camada oculta (entrada = tamanho do estado, saída = `h_size`), com ativação **ReLU**
- `fc2`: camada de saída (entrada = `h_size`, saída = número de ações), seguida de **softmax** — a saída é uma **distribuição de probabilidades sobre as ações**

A função `act` recebe o estado, roda a rede e **amostra uma ação** da distribuição, devolvendo também o **log da probabilidade** da ação escolhida (usado depois no cálculo do gradiente).

✏️ **Exercício**: complete os espaços e depois confira na solução.

In [ ]:
# Exercício: complete os espaços e descomente.
# class Policy(nn.Module):
#     def __init__(self, s_size, a_size, h_size):
#         super(Policy, self).__init__()
#         self.fc1 = nn.Linear(s_size, ???)
#         self.fc2 = nn.Linear(???, a_size)
#
#     def forward(self, x):
#         x = F.relu(self.fc1(x))
#         x = self.fc2(x)
#         return F.???(x, dim=1)
#
#     def act(self, state):
#         state = torch.from_numpy(state).float().unsqueeze(0).to(device)
#         probs = self.forward(state).cpu()
#         m = Categorical(probs)
#         action = m.sample()
#         return action.item(), m.log_prob(action)

<details><summary><b>Solução</b></summary>

```python
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)
```
</details>

In [ ]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

## O algoritmo REINFORCE 🔥

Para cada episódio:

1. Coletamos trajetórias amostrando ações da política e guardamos `(log π(a|s), recompensa)` de cada passo.
2. Calculamos o **retorno descontado** `G_t` de cada passo: `G_t = r_(t+1) + γ·G_(t+1)` (somando do fim para o começo, em O(n)).
3. **Normalizamos** os retornos (subtrair a média e dividir pelo desvio-padrão) — isso deixa o treino mais estável.
4. Calculamos a perda `L = −Σ G_t · log π(a_t|s_t)` e fazemos um passo do otimizador. Minimizar a perda negativa = **maximizar** a recompensa esperada (gradiente ascendente).

✏️ **Exercício**: complete a célula do treino (o cálculo do retorno e a perda).

In [ ]:
# Exercício: complete os espaços e descomente.
# def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
#     scores_deque = deque(maxlen=100)
#     scores = []
#
#     for i_episode in range(1, n_training_episodes + 1):
#         saved_log_probs = []
#         rewards = []
#         state, info = env.reset()
#
#         for t in range(max_t):
#             action, log_prob = policy.act(state)
#             saved_log_probs.append(log_prob)
#             state, reward, terminated, truncated, info = env.step(action)
#             rewards.append(reward)
#             if terminated or truncated:
#                 break
#
#         scores_deque.append(sum(rewards))
#         scores.append(sum(rewards))
#
#         # Calcula os retornos descontados, do último passo para o primeiro:
#         #   G_t = r_(t+1) + gamma * G_(t+1)
#         returns = deque(maxlen=max_t)
#         n_steps = len(rewards)
#         for t in range(n_steps)[::-1]:
#             disc_return_t = returns[0] if len(returns) > 0 else 0
#             returns.appendleft(???)
#
#         # Normalização dos retornos (estabiliza o treino)
#         eps = np.finfo(np.float32).eps.item()
#         returns = torch.tensor(returns)
#         returns = (returns - returns.mean()) / (returns.std() + eps)
#
#         # Perda: -sum(G_t * log_prob(a_t|s_t))
#         policy_loss = []
#         for log_prob, disc_return in zip(saved_log_probs, returns):
#             policy_loss.append(???)
#         policy_loss = torch.cat(policy_loss).sum()
#
#         optimizer.zero_grad()
#         policy_loss.backward()
#         optimizer.step()
#
#         if i_episode % print_every == 0:
#             print(f"Episódio {i_episode}\tPontuação média: {np.mean(scores_deque):.2f}")
#
#     return scores

<details><summary><b>Solução</b></summary>

```python
        for t in range(n_steps)[::-1]:
            disc_return_t = returns[0] if len(returns) > 0 else 0
            returns.appendleft(gamma * disc_return_t + rewards[t])
```

```python
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
```
</details>

In [ ]:
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # Fila para acompanhar a pontuação média dos últimos 100 episódios
    scores_deque = deque(maxlen=100)
    scores = []

    for i_episode in range(1, n_training_episodes + 1):
        saved_log_probs = []
        rewards = []
        state, info = env.reset()

        # Coleta a trajetória do episódio
        for t in range(max_t):
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            state, reward, terminated, truncated, info = env.step(action)
            rewards.append(reward)
            if terminated or truncated:
                break

        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Retornos descontados, do último passo para o primeiro:
        #   G_t = r_(t+1) + gamma * G_(t+1)
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        for t in range(n_steps)[::-1]:
            disc_return_t = returns[0] if len(returns) > 0 else 0
            returns.appendleft(gamma * disc_return_t + rewards[t])

        # Normalização dos retornos (estabiliza o treino)
        eps = np.finfo(np.float32).eps.item()
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # Perda: -sum(G_t * log_prob(a_t|s_t))
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print(f"Episódio {i_episode}\tPontuação média: {np.mean(scores_deque):.2f}")

    return scores

## Treinar o agente 🏃

Definimos os hiperparâmetros e treinamos. Com 1000 episódios (cada um com no máximo 500 passos), o treino leva poucos minutos no Colab.

💡 Para experimentar, você pode mexer em `gamma`, `lr` (taxa de aprendizado) e `h_size` (tamanho da camada oculta).

In [ ]:
cartpole_hyperparameters = {
    "h_size": 16,
    "n_training_episodes": 1000,
    "n_evaluation_episodes": 10,
    "max_t": 1000,
    "gamma": 1.0,
    "lr": 1e-2,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

In [ ]:
# Cria a política e o otimizador
cartpole_policy = Policy(cartpole_hyperparameters["state_space"],
                         cartpole_hyperparameters["action_space"],
                         cartpole_hyperparameters["h_size"]).to(device)
cartpole_optimizer = optim.Adam(cartpole_policy.parameters(),
                                lr=cartpole_hyperparameters["lr"])

In [ ]:
scores = reinforce(cartpole_policy,
                   cartpole_optimizer,
                   cartpole_hyperparameters["n_training_episodes"],
                   cartpole_hyperparameters["max_t"],
                   cartpole_hyperparameters["gamma"],
                   100)

# Curva de aprendizado: pontuação de cada episódio
plt.plot(scores)
plt.xlabel("Episódio")
plt.ylabel("Pontuação")
plt.title("Aprendizado do REINFORCE no CartPole-v1")
plt.show()

## Avaliar o agente 📈

A avaliação roda `n_eval_episodes` episódios com a política treinada (sem atualizar os pesos) e devolve a **recompensa média e o desvio-padrão**.

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, policy):
    """
    Avalia o agente por ``n_eval_episodes`` episódios e devolve
    a recompensa média e o desvio-padrão.
    """
    episode_rewards = []
    for episode in range(n_eval_episodes):
        state, info = env.reset()
        total_rewards_ep = 0

        for step in range(max_steps):
            action, _ = policy.act(state)
            new_state, reward, terminated, truncated, info = env.step(action)
            total_rewards_ep += reward
            if terminated or truncated:
                break
            state = new_state

        episode_rewards.append(total_rewards_ep)

    return np.mean(episode_rewards), np.std(episode_rewards)

In [ ]:
evaluate_agent(eval_env,
               cartpole_hyperparameters["max_t"],
               cartpole_hyperparameters["n_evaluation_episodes"],
               cartpole_policy)

## Gravar um vídeo do replay 🎥

A função `record_video` (adaptada do 0_qlearning.ipynb) cria um ambiente com `render_mode="rgb_array"` e grava a política jogando.

In [ ]:
def record_video(env_id, policy, out_directory, fps=30, max_steps=500):
    """
    Gera um vídeo de replay do agente treinado
    :param env_id: id do ambiente (ex.: "CartPole-v1")
    :param policy: a política treinada
    :param out_directory: arquivo de saída (ex.: "replay-cartpole.mp4")
    :param fps: quadros por segundo no vídeo
    :param max_steps: número máximo de passos a gravar
    """
    import os
    os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # necessário para renderizar no Colab

    env = gym.make(env_id, render_mode="rgb_array")

    images = []
    state, info = env.reset()
    images.append(env.render())

    for _ in range(max_steps):
        action, _ = policy.act(state)
        state, reward, terminated, truncated, info = env.step(action)
        images.append(env.render())
        if terminated or truncated:
            break

    env.close()
    imageio.mimsave(out_directory, [np.array(img) for img in images], fps=fps)
    print(f"Vídeo salvo em {out_directory} ({len(images)} quadros)")

In [ ]:
# Gravar um vídeo do resultado do treino
video_path = "replay-cartpole.mp4"
record_video(env_id, cartpole_policy, video_path)

## Alguns desafios adicionais 🏆

A melhor maneira de aprender **é tentar as coisas por conta própria**!

- Experimente `gamma` em {0.95, 0.99, 1.0} — o que muda na curva de aprendizado?
- Tente outros valores de `lr` e `h_size` — consegue treinar mais rápido?
- Compare com os resultados dos seus colegas: o objetivo é chegar a uma recompensa média de avaliação próxima de **500** (o máximo do CartPole-v1).

*(O notebook original do curso também treinava um segundo agente no jogo PixelCopter; deixamos essa parte de fora porque a biblioteca dele (gym_pygame) é incompatível com o gymnasium atual.)*

________________________________________________________________________
Parabéns por terminar esta atividade! 🥳

### Continue Aprendendo, Continue Incrível 🤗